# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Lane 2 — Refresh / Content Opportunity Scoring**

**Primary task type: ranking / scoring.** The deliverable is a ranked review queue — editors need "which pages first," not just yes/no on one page. The score orders candidates under limited capacity.

**Supporting task type: binary classification.** To rank pages, I first estimate review priority from observable signals. On the starter slice, that means a probability that a page matches a decline/opportunity proxy (`is_declining_label`). That probability feeds a combined score; the list is sorted by score.

**Not clustering or pure signal analysis.** Clustering would describe page types without a priority order. Signal analysis would stop at associations. My lane needs an ordered shortlist tied to editorial actions (refresh, expand, CTR review, engagement review, monitor).

**ML loop mapping:** define unit (page) → define proxy/target → build transparent baseline score → train classifier for priority probability → rank by score → evaluate with Precision@K under client holdout → export queue with reason codes.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Starter proxy target (what I can build today):** `is_declining_label` — 1 if `trend_direction == "down"`, else 0. Same definition as `scripts/01_prepare_features.py`. This is a **defined rule from the current 90-day window**, not a future observed outcome. I treat it as a teaching proxy that matches the starter pipeline.

**Capstone target (stronger, later):** a **future-window observed outcome** built from warehouse daily facts — for example, features from prior 90 days → decline or recovery measured in the next 30 days. That label is observed after a clear decision point and avoids circular "predict the bucket we already computed."

**What I will never use as features:** `trend_direction` and `trend_pct` — they define (or strongly encode) the starter proxy label, so using them as inputs would be leakage.

**Ranking output column (what the editor sees):** `final_refresh_score` — a 0–100 priority score combining model probability and baseline score. The **action** comes from reason codes (e.g. refresh, refresh_and_review_ctr, monitor) attached to each ranked row.

**Label source honesty:** starter proxy = rule-defined; capstone = observed future movement. I will say which one a result used.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Primary metric: Precision@50**

**What it measures:** Of the top 50 pages my ranked queue recommends first, what fraction actually match the proxy target (`is_declining_label == 1`)?

**Why this metric:** Editors have limited weekly capacity — often on the order of 20–50 reviews. Precision@K matches how the list is actually used: "If I only open the top K pages, how many were worth my time?" Generic accuracy is misleading here because ~54% of pages are declining on the starter proxy — a dumb list could look okay without helping prioritization.

**What "good" looks like (starter benchmark, client-holdout):**
- Transparent baseline rules: Precision@50 ≈ **0.24** (~12 of 50)
- Random forest (starter pipeline): Precision@50 ≈ **0.74** (~37 of 50)

**Secondary metrics (supporting, not primary):** average precision (whole ranking quality), ROC-AUC (separation across all thresholds). I will report these but choose actions using Precision@K because that matches reviewer capacity.

**Validation design:** client-grouped holdout — whole clients held out of training so the model is tested on clients it never saw. That matches the real decision: prioritize pages for clients the system has not memorized.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [1]:
from pathlib import Path

import pandas as pd

# --- Load starter data (repo-relative paths) ---
candidates = [
    Path("../../data/raw/content_refresh_anonymized.csv"),
    Path("data/raw/content_refresh_anonymized.csv"),
]
raw_path = next(p for p in candidates if p.exists())
raw = pd.read_csv(raw_path)

# --- Lane 2 slice: visible pages old enough to review (same filters as starter pipeline) ---
lane = raw[(raw["impressions_90d"] > 0) & (raw["content_age_days"] >= 90)].copy()
lane = lane.drop_duplicates(subset=["content_id"]).reset_index(drop=True)

# --- Target / proxy columns (starter now; capstone later) ---
lane["is_declining_label"] = lane["trend_direction"].str.lower().eq("down").astype(int)

# Sketch what a future-window capstone label would look like (not computable from this snapshot alone)
lane["future_decline_30d"] = pd.NA  # placeholder: 1/0 from warehouse daily facts, next 30 days
lane["review_priority_score"] = pd.NA  # placeholder: filled after baseline + model scoring

# --- Unit-of-analysis view: one row = one content page ---
unit_cols = [
    "content_id",
    "client_id",
    "impressions_90d",
    "sessions_90d",
    "content_age_days",
    "days_since_last_update",
    "avg_position",
    "ctr",
    "trend_direction",
    "is_declining_label",
    "future_decline_30d",
    "review_priority_score",
]
unit_df = lane[unit_cols].copy()

print("Unit of analysis: one row = one content page (content_id)")
print(f"Lane slice rows: {len(unit_df):,} pages across {unit_df['client_id'].nunique()} clients")
print(f"Grain check — duplicate content_id rows: {unit_df['content_id'].duplicated().sum()}")
print()
print("Target / proxy distribution on starter slice:")
print(unit_df["is_declining_label"].value_counts().rename({0: "not_declining_proxy", 1: "declining_proxy"}))
print(f"Base rate (declining proxy): {unit_df['is_declining_label'].mean():.1%}")
print()
print("Sample rows (unit + target sketch):")
print(unit_df.head(8).to_string(index=False))

# Tie metric to real numbers: how many proxy-positives compete for top-K slots?
positives = int(unit_df["is_declining_label"].sum())
print()
print(f"Proxy-positive pages in lane slice: {positives:,}")
print(
    "If an editor reviews 50 pages/week, Precision@50 asks: "
    "how many of those 50 are truly proxy-positive — not whether we found all "
    f"{positives:,} positives."
)

Unit of analysis: one row = one content page (content_id)
Lane slice rows: 30,000 pages across 32 clients
Grain check — duplicate content_id rows: 0

Target / proxy distribution on starter slice:
is_declining_label
declining_proxy        16262
not_declining_proxy    13738
Name: count, dtype: int64
Base rate (declining proxy): 54.2%

Sample rows (unit + target sketch):
          content_id         client_id  impressions_90d  sessions_90d  content_age_days  days_since_last_update  avg_position  ctr trend_direction  is_declining_label future_decline_30d review_priority_score
content_304f48230142 client_f369cb89fc             3803            17               187                      20          10.6 0.76            down                   1               <NA>                  <NA>
content_a1fb4e703a9e client_4e07408562            15320             9               445                      25          20.3 0.05            down                   1               <NA>                  <NA>
conte

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule already helps — the starter baseline uses readable if-statements like *declining + demand*, *stale + visible*, *low CTR at a given position*, and *thin visible page*. I would build that baseline first and beat it honestly, or keep it.

**Why a plain rule is not enough here:**
1. **Scale:** ~13k declining-with-demand pages in the starter slice, but an editor may only review 50/week. Rules flag many candidates; something must **rank** within that flagged set.
2. **Tangled signals:** Volume, age, freshness, position, CTR, engagement, and depth interact. A page can be declining, high-impression, page-one, low-CTR, and stale at once — fixed weights (40/30/25/05 in the starter baseline) are a starting point, not proof they are optimal for every client.
3. **Evidence it matters:** On the starter slice with client holdout, the baseline hit Precision@50 ≈ 0.24 while a random forest hit ≈ 0.74 on the same proxy label. That is directional evidence that learned scoring can reorder candidates better than hand-tuned rules — on this slice, with this proxy.
4. **Action tie-in:** The output is not "declining = yes." It is a **ranked queue with reason codes and suggested actions** (refresh, expand, CTR review, engagement review, monitor). ML earns its place if it puts the highest-value review candidates at the top of that list.

**Careful limit:** Better ranking on a proxy label does not prove a refresh causes recovery. The safe claim is decision-support — help editors spend scarce review time on the most promising pages first.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.